## ML_1020: Grouped Query Attention (GQA)

**Difficulty**: Hard | **TorchCode**: #10

### Problem
Implement Grouped Query Attention (LLaMA 2-style). Use fewer KV heads than Q heads: each group of Q heads shares one K/V head. Expand KV heads using `repeat_interleave`.

### Formula
$$\text{groups} = h_Q / h_{KV}, \quad K_{\text{expanded}} = \text{repeat\_interleave}(K,\, \text{groups},\, \text{dim}=1)$$

In [ ]:
import torch
import torch.nn as nn
import math

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, S, _ = x.shape
        q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)
        repeats = self.num_heads // self.num_kv_heads
        k = k.repeat_interleave(repeats, dim=1)
        v = v.repeat_interleave(repeats, dim=1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        weights = torch.softmax(scores, dim=-1)
        attn = torch.matmul(weights, v)
        out = attn.transpose(1, 2).contiguous().view(B, S, -1)
        return self.W_o(out)

In [ ]:
import torch
import torch.nn as nn

# --- Test 1: Output shape ---
torch.manual_seed(0)
gqa = GroupQueryAttention(32, 8, 2)
assert gqa.forward(torch.randn(2, 6, 32)).shape == (2, 6, 32)

# --- Test 2: nn.Linear shapes ---
d_k = 32 // 8
assert isinstance(gqa.W_q, nn.Linear) and gqa.W_q.weight.shape == (32, 32)
assert isinstance(gqa.W_k, nn.Linear) and gqa.W_k.weight.shape == (2 * d_k, 32)
assert isinstance(gqa.W_v, nn.Linear) and gqa.W_v.weight.shape == (2 * d_k, 32)

# --- Test 3: Degenerates to MHA when kv_heads == heads ---
torch.manual_seed(42)
gqa_mha = GroupQueryAttention(16, 4, 4)
assert gqa_mha.forward(torch.randn(1, 4, 16)).shape == (1, 4, 16)
assert gqa_mha.W_k.weight.shape == (16, 16)

# --- Test 4: KV heads shared correctly ---
torch.manual_seed(0)
D, H, KV = 16, 4, 2; d_k2 = D // H
gqa2 = GroupQueryAttention(D, H, KV)
x = torch.randn(1, 4, D)
k = gqa2.W_k(x).view(1, 4, KV, d_k2).transpose(1, 2)
k_exp = k.repeat_interleave(H // KV, dim=1)
assert torch.equal(k_exp[:, 0], k_exp[:, 1])
assert not torch.equal(k_exp[:, 0], k_exp[:, 2])

# --- Test 5: Gradient flow ---
torch.manual_seed(0)
gqa3 = GroupQueryAttention(16, 4, 2)
x = torch.randn(1, 4, 16, requires_grad=True)
gqa3.forward(x).sum().backward()
assert x.grad is not None

print('All tests passed!')